# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [ ]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Setup complete!



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Tanushka\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: False


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [11]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence
llm_X = vec.transform(llm["review"])
llm_probs = clf.predict_proba(llm_X)
llm_confidence = llm_probs.max(axis=1)
llm_predictions = clf.predict(llm_X)

filtered_llm = llm[
    (llm_confidence >= 0.65) & (llm_predictions == llm["label"])
].copy()
print(f"Filtered LLM samples (conf >= 0.65 & agree): {len(filtered_llm)}")

#  1d. Merge & deduplicate
consolidated = pd.concat([gold, weak, filtered_llm], ignore_index=True)
consolidated = consolidated.drop_duplicates(subset="review").reset_index(drop=True)
consolidated.to_csv("consolidated_base.csv", index=False)
print(f"consolidated_base.csv saved: {len(consolidated)} unique reviews")

#  1e. Class distribution analysis
class_counts = consolidated["label"].value_counts()
minority_class = class_counts.idxmin()
print(f"Class distribution:\n{class_counts}")
print(f"Minority class: {minority_class}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(class_counts.index, class_counts.values,
              color=["#4CAF50", "#F44336", "#2196F3"][:len(class_counts)])
ax.set_title("Class Distribution of Consolidated Dataset", fontsize=14)
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.close()
print("class_distribution.png saved.")

Gold: 100, Weak: 220, LLM: 150
Filtered LLM samples (conf >= 0.65 & agree): 27
consolidated_base.csv saved: 328 unique reviews
Class distribution:
label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64
Minority class: Positive
class_distribution.png saved.


In [12]:

#  1f. Augmentation functions

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    # Map Penn POS tags to WordNet POS tags
    def get_wn_pos(tag):
        if tag.startswith("J"):
            return wordnet.ADJ
        elif tag.startswith("V"):
            return wordnet.VERB
        elif tag.startswith("N"):
            return wordnet.NOUN
        elif tag.startswith("R"):
            return wordnet.ADV
        return None

    # Identify replaceable indices: not preserve words, not punctuation
    replaceable = [
        i for i, (word, tag) in enumerate(tagged)
        if word.lower() not in PRESERVE_WORDS
        and word.isalpha()
        and get_wn_pos(tag) is not None
    ]

    n_replace = max(1, int(len(replaceable) * replace_frac))
    indices_to_replace = random.sample(replaceable, min(n_replace, len(replaceable)))

    new_tokens = tokens[:]
    for i in indices_to_replace:
        word, tag = tagged[i]
        wn_pos = get_wn_pos(tag)
        synsets = wordnet.synsets(word, pos=wn_pos)
        synonyms = []
        for syn in synsets:
            for lemma in syn.lemmas():
                candidate = lemma.name().replace("_", " ")
                if candidate.lower() != word.lower() and candidate.isalpha():
                    synonyms.append(candidate)
        if synonyms:
            new_tokens[i] = random.choice(synonyms)

    return " ".join(new_tokens)


def back_translate(text, src="en", mid="hi"):
    """Translate English → Hindi → English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    try:
        hindi = GoogleTranslator(source=src, target=mid).translate(text)
        time.sleep(0.5)  # rate-limit sleep
        back = GoogleTranslator(source=mid, target=src).translate(hindi)
        time.sleep(0.5)
        return back if back else text
    except Exception as e:
        print(f"Back-translation failed: {e}")
        return text


def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    set_orig = set(original.lower().split())
    set_aug  = set(augmented.lower().split())
    union = set_orig | set_aug
    if not union:
        return False
    jaccard = len(set_orig & set_aug) / len(union)
    return 0.30 <= jaccard <= 0.95


#  1g. Apply augmentation to minority class
minority_samples = consolidated[consolidated["label"] == minority_class].copy()
print(f"Minority class '{minority_class}' has {len(minority_samples)} samples. Augmenting...")

augmented_rows = []
for _, row in minority_samples.iterrows():
    original_text = row["review"]
    label = row["label"]

    # Method 1: Synonym Replacement
    syn_aug = synonym_replacement(original_text, replace_frac=random.uniform(0.15, 0.20))
    if syn_aug and quality_filter(original_text, syn_aug):
        augmented_rows.append({"review": syn_aug, "label": label, "augmentation": "synonym_replacement"})

    # Method 2: Back Translation
    bt_aug = back_translate(original_text)
    if bt_aug and quality_filter(original_text, bt_aug):
        augmented_rows.append({"review": bt_aug, "label": label, "augmentation": "back_translation"})

augmented_df = pd.DataFrame(augmented_rows)
print(f"Augmented samples generated (after quality filter): {len(augmented_df)}")

# Combine original consolidated with augmented minority samples
augmented_classical = pd.concat(
    [consolidated.assign(augmentation="original"), augmented_df],
    ignore_index=True
)
augmented_classical = augmented_classical.drop_duplicates(subset="review").reset_index(drop=True)
augmented_classical.to_csv("augmented_classical.csv", index=False)
print(f"augmented_classical.csv saved: {len(augmented_classical)} total rows")

Minority class 'Positive' has 62 samples. Augmenting...
Augmented samples generated (after quality filter): 123
augmented_classical.csv saved: 449 total rows


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt 
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
PROMPT_TEMPLATE = """You are a movie review generator. ..."""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches 
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors


#  2c. Diversity metrics 
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score


#  2d. Sentiment consistency check 
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv


#  2e. Save outputs 
# TODO: Save llm_generated_300.csv and diversity_report.txt

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [ ]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling 
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)


#  3b. Translation pipeline 
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits


#  3c. Back-translation & BLEU 
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"


#  3d. Sentiment preservation check 
# TODO: Predict sentiment on back-translated text, compare with original label


#  3e. Manual verification 
# TODO: Print 5 random samples for inspection


#  3f. Save 
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [9]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class) 
# TODO: Sample from consolidated_base, mix short and long reviews


#  4b. TTS generation 
os.makedirs("audio_samples", exist_ok=True)

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile


#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv


#  4d. Whisper round-trip validation 
import whisper

_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv

100%|█████████████████████████████████████| 72.1M/72.1M [00:44<00:00, 1.70MiB/s]


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [10]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv


#  5b. Black-Box Evaluation 
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)

# Print comparison
# print(f"Baseline accuracy:  {baseline_acc:.2%}")
# print(f"Augmented accuracy: {augmented_acc:.2%}")
# print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")

Initializing Black-Box Embedder...


ValueError: The provided filename text_embedder.pt does not exist